# Build the package from these notebooks

Edit a source cell, run its nearby experiment, save, then export. The generated Python files are the installable package; these notebooks are their source.

**Development notebook.** Expand the library cells to edit the implementation. Examples below use local data and CPU. [Build workflow](README.md#develop-the-package-here).

In [1]:
#| default_exp __init__

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'pyproject.toml').is_file() and (p / 'timegym').is_dir())
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 4), 'figure.dpi': 110,
    'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'legend.frameon': False, 'axes.titlelocation': 'left'})

In [3]:
#| export _internal
from dataclasses import fields, is_dataclass
from enum import Enum
import math
import traceback
from typing import Any
import numpy as np


def _integer(value: Any, name: str, *, minimum: int) -> int:
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < minimum:
        qualifier = "positive" if minimum else "nonnegative"
        raise ValueError(f"{name} must be a {qualifier} integer")
    return int(value)


def positive_int(value: Any, name: str) -> int:
    return _integer(value, name, minimum=1)


def nonnegative_int(value: Any, name: str) -> int:
    return _integer(value, name, minimum=0)

In [4]:
#| export _internal
def _has_complex_values(array: np.ndarray) -> bool:
    """Detect complex scalars even when an object array hides their dtype."""
    return np.iscomplexobj(array) or (
        array.dtype.kind == "O" and any(np.iscomplexobj(value) for value in array.flat)
    )

In [5]:
#| export _internal
def _real_array(value: Any, name: str) -> np.ndarray:
    """Convert real numeric data without discarding an imaginary component."""
    array = np.asarray(value)
    if _has_complex_values(array):
        raise ValueError(f"{name} must contain real-valued numbers")
    return np.asarray(array, dtype=float)

In [6]:
#| export _internal
def _exception_evidence(exc: BaseException) -> dict:
    """Format the supplied exception even outside its handling block."""
    return {
        "exception_type": type(exc).__name__,
        "exception_message": str(exc),
        "traceback": "".join(traceback.format_exception(exc)),
    }


def _json_safe(value):
    """Canonicalize evidence to JSON values, rejecting nonfinite numbers."""
    if isinstance(value, Enum):
        return _json_safe(value.value)
    if value is None:
        return None
    if isinstance(value, str):
        return str(value)
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if not math.isfinite(value):
            raise ValueError("JSON evidence contains a nonfinite number")
        return float(value)
    if isinstance(value, np.generic):
        return _json_safe(value.item())
    if isinstance(value, np.ndarray):
        return _json_safe(value.tolist())
    if isinstance(value, BaseException):
        return _exception_evidence(value)
    if is_dataclass(value):
        return {item.name: _json_safe(getattr(value, item.name)) for item in fields(value)}
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [_json_safe(item) for item in value]
    if hasattr(value, "isoformat"):
        return str(value.isoformat())
    return repr(value)


## Build `__init__`

### Module definitions

In [7]:
#| export __init__
from timegym.contracts import Forecast, ForecastTask, Reference, SimulationReference
from timegym.scenarios import Scenario, Case
from timegym.pipeline import Pipeline
from timegym.models import NaiveForecaster, GaussianAR1Forecaster
from timegym.reporting import TestReport, DiagnosticStatus, DiagnosticCategory, DiagnosticResult, CaseResult
from timegym.evaluation import ForecastRun, EvaluatedCase

__version__ = "0.2.0"
__all__ = ["Case", "Forecast", "ForecastTask", "Reference", "SimulationReference", "Scenario", "Pipeline",
           "NaiveForecaster", "GaussianAR1Forecaster", "TestReport", "ForecastRun", "EvaluatedCase", "DiagnosticStatus", "DiagnosticCategory", "DiagnosticResult", "CaseResult"]

## Inspect the public entry point

The root imports expose forecasting and evaluation. Data-preparation helpers are
also public, through `timegym.features`, `timegym.target`, `timegym.smoothing`,
and `timegym.data.real`; use explicit imports from those modules. Their contracts
and optional dependencies are documented in the [API reference](../docs/reference.md).


In [8]:
# Inspect the entry point defined here, not a previously imported copy.
assert all(name in globals() for name in __all__)
print('TimeGym', __version__)
display(pd.DataFrame({'public_name': __all__}))

TimeGym 0.2.0


,public_name
0,Case
1,Forecast
2,ForecastTask
3,Reference
4,SimulationReference
5,Scenario
6,Pipeline
7,NaiveForecaster
8,GaussianAR1Forecaster
9,TestReport


From the repository root run `python tools/export_notebooks.py`, then `python -m pytest`. `python tools/export_notebooks.py --check` detects source drift without writing. `python notebooks/check_notebooks.py` executes every active chapter in a fresh kernel. Export never runs demonstration cells, and ordinary Run All never rewrites package modules.

## Edit, run, export

After editing a source cell, **restart this kernel and Run All**. This refreshes the local definitions, the development bindings between modules in this chapter, and the nearby examples. Ordinary Run All does not export files.

Changes to code owned by another chapter require saving that source notebook, running `python tools/export_notebooks.py`, and restarting this kernel before importing the updated package. Runtime provenance describes the exported package, not unexported edits.

Try one implementation or parameter change, rerun this chapter, and inspect the figure or assertion. Save the notebook before exporting.